<a href="https://colab.research.google.com/github/Gan4x4/cv/blob/main/Segmentation/SMP.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Segmentation models PyTorch (SMP)

[[git] 🐾 Библиотека](https://github.com/qubvel/segmentation_models.pytorch#architectures-) на базе PyTorch  для сегментации.

<img src ="https://ml.gan4x4.ru/msu/dep-2.1/L11/smp.png" width="1000">



In [ ]:
from IPython.display import clear_output

!pip install -q segmentation-models-pytorch
clear_output()

Можем комбинировать декодер с разными энкодерами:

In [ ]:
import torch
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

# 'mit_b0' Mix Vision Transformer Backbone from SegFormer pretrained on Imagenet
preprocess_input = get_preprocessing_fn("mit_b0", pretrained="imagenet")

# MixVisionTransformer encoder does not support in_channels setting other than 3
# supported by FPN only for encoder depth = 5
model = smp.FPN("mit_b0", in_channels=3, classes=10, encoder_depth=5)

# ... Train model on your dataset

dummy_input = torch.randn([1, 3, 64, 64])

mask = model(dummy_input)
clear_output()
print(mask.shape)  # torch.Size([1, 1, 64, 64])

### Совместимость с timm

Существует библиотека [pytorch-image-models 🐾[git]](https://github.com/huggingface/pytorch-image-models) (timm = Torch IMage Models), в которой собрано большое количество моделей для работы с изображениями.

[Описание библиотеки 🛠️[doc]](https://huggingface.co/docs/timm/index) и [примеры использования 🛠️[doc]](https://huggingface.co/docs/hub/timm) в Hugging Face.


In [ ]:
import timm

model_names = timm.list_models(pretrained=True)
print("Total pretrained models: ", len(model_names))

Можно искать модели по шаблону:

In [ ]:
model_names = timm.list_models("*mobilenet*small*")
print(model_names)

Smoke test:

In [ ]:
timm_mobilenet = timm.create_model("mobilenetv3_small_050", pretrained=True)
clear_output()

In [ ]:
out = timm_mobilenet(dummy_input)
print(out.shape)

Можно использовать большинство моделей из timm в качестве энкодеров.

[[doc] 🛠️ Список совместимых моделей](https://smp.readthedocs.io/en/latest/encoders_timm.html)

При этом к названию модели, которое передается в конструктор класса SMP, нужно добавить префикс `tu-`:

In [ ]:
smp_timm_model = smp.DeepLabV3("tu-mobilenetv3_small_050", in_channels=3, classes=80)
smp_timm_model.eval()
print("Created DeepLab with mobileNet encoder")

Smoke test:

In [ ]:
mask = smp_timm_model(dummy_input)
print(mask.shape)